# 🏭 MLOps & Production Serving: Zero to Hero — A Guided Lab

Training a great model is half the job. This lab covers the other half: **serving** models
reliably, **monitoring** them in production, catching **drift** before it hurts users, and
managing the **cost/latency** tradeoffs that make or break a real deployment.

**Beginner-first.** Every chapter explains the *concept* before code. Prerequisite: the
Evaluation & Guardrails lab (eval sets, regression testing) and any of the model-training labs.

**How this lab works** — 📖 Theory → 🧠 Mental model → 🖼️ ASCII diagram → 🔬 Worked example →
⚡ Pro tips → ⚠️ Traps → ✏️ Your Turn → ✅ Solution.

**Roadmap**
1. From notebook to production: what changes
2. Model serving patterns (batch vs. real-time vs. streaming)
3. Latency & throughput budgets
4. Caching strategies
5. Monitoring: what to track and why
6. Data drift & model drift
7. A/B testing & canary deployments
8. Logging & observability for LLM apps
9. Cost management at scale
10. 🏆 Capstone: a monitored, guarded serving layer


In [ ]:
import time, random, json
from collections import deque, Counter
print("Ready.")

---
## Chapter 1 — From Notebook to Production: What Changes

📖 **Theory.** A notebook that works once is very different from a system that must work
**reliably, thousands of times a day, without you watching it**. Production adds requirements a
notebook never needs to think about:
- **Reliability** — handle errors gracefully instead of crashing the kernel.
- **Scale** — serve many concurrent requests, not just one at a time.
- **Observability** — you need to know *what happened* without re-running code by hand.
- **Change management** — deploying a new model version safely, without breaking users.

🖼️ **Diagram — notebook vs. production**
```
 NOTEBOOK:  run cell -> see output -> fix bug -> re-run    (you're always watching)

 PRODUCTION: request ─► model ─► response    (happens thousands of times, unattended)
                  │                  │
              logged            monitored           <- you find out about problems from DATA, not by watching
```

🧠 **Mental model.** In a notebook, *you* are the error handler and the monitor. In production,
the **system** has to be its own error handler and monitor — because you won't be there when
something breaks at 3am.


In [ ]:
# A notebook-style function -- works, but fragile
def notebook_predict(model_fn, input_data):
    return model_fn(input_data)   # if model_fn throws, this crashes -- fine in a notebook, not in prod

# A production-style wrapper -- same core logic, but defensive
def production_predict(model_fn, input_data, fallback=None):
    try:
        start = time.time()
        result = model_fn(input_data)
        elapsed = time.time() - start
        return {"ok": True, "result": result, "latency_ms": round(elapsed*1000, 2)}
    except Exception as e:
        return {"ok": False, "error": str(e), "fallback": fallback}

def flaky_model(x):
    if x < 0: raise ValueError("negative input not supported")
    return x * 2

print(production_predict(flaky_model, 5))
print(production_predict(flaky_model, -3, fallback=0))

### ✏️ Your Turn 1.1
In a comment, list two things a production wrapper needs that a plain notebook function
doesn't.

In [ ]:
# 1. ...
# 2. ...


✅ **Solution**
```python
# 1. Error handling that returns a structured failure instead of crashing the whole process.
# 2. Timing/logging so you can measure latency and diagnose issues after the fact,
#    without being able to just "watch it run" like in a notebook.
```

---
## Chapter 2 — Model Serving Patterns

📖 **Theory.** Three main patterns, chosen by **how urgently** a response is needed:
- **Batch** — process a large set of inputs on a schedule (e.g. nightly), no real-time
  constraint. Cheapest, simplest.
- **Real-time (online)** — respond to individual requests as they arrive, usually with a strict
  latency budget (e.g. a chat UI waiting on a response).
- **Streaming** — continuously process an unbounded stream of events (e.g. tokens as an LLM
  generates them, or live sensor data).

🖼️ **Diagram — the three patterns**
```
 BATCH:       [1000s of inputs] ──(runs overnight)──► [1000s of outputs, all at once]
 REAL-TIME:   1 request ──(must respond in <1s)──► 1 response
 STREAMING:   ──event──event──event──event──►  (continuous, processed as they arrive)
```


In [ ]:
def batch_predict(model_fn, inputs):
    """Batch: process everything, return all results together. No per-item latency pressure."""
    return [model_fn(x) for x in inputs]

def realtime_predict(model_fn, single_input, timeout_ms=500):
    """Real-time: must respond fast, single input at a time, timeout enforced."""
    start = time.time()
    result = model_fn(single_input)
    elapsed_ms = (time.time() - start) * 1000
    if elapsed_ms > timeout_ms:
        return {"ok": False, "error": "timeout_exceeded", "elapsed_ms": elapsed_ms}
    return {"ok": True, "result": result, "elapsed_ms": round(elapsed_ms, 2)}

def simple_model(x): return x ** 2

print("batch:", batch_predict(simple_model, [1,2,3,4,5]))
print("real-time:", realtime_predict(simple_model, 7, timeout_ms=100))

⚡ **Pro tip.** Choose the **cheapest** pattern that meets your actual latency need. Batch is
far simpler and cheaper to operate than real-time — don't build a real-time system for a report
that's only viewed once a day.

### ✏️ Your Turn 2.1
Decide which pattern (batch/real-time/streaming) fits each: (a) generating a weekly sales report,
(b) a customer support chatbot, (c) live transcription of a phone call.

In [ ]:
# (a) ...
# (b) ...
# (c) ...


✅ **Solution**
```python
# (a) batch      -- no urgency, runs on a schedule
# (b) real-time  -- user is waiting for a response right now
# (c) streaming  -- continuous unbounded audio needs continuous processing
```

---
## Chapter 3 — Latency & Throughput Budgets

📖 **Theory.** Two related but different metrics:
- **Latency** — how long **one** request takes (measured in percentiles: p50, p95, p99 — the
  99th percentile matters more than the average, since it captures your worst user experiences).
- **Throughput** — how many requests you can handle **per second**, in aggregate.

🖼️ **Diagram — why percentiles matter more than averages**
```
 100 requests: 95 take 50ms, 5 take 2000ms (a bad tail)
 average latency: ~148ms   <- looks okay!
 p99 latency: ~2000ms      <- reveals the real problem 1 in 20 users experience
```


In [ ]:
def measure_latencies(model_fn, inputs):
    latencies = []
    for x in inputs:
        start = time.time()
        model_fn(x)
        latencies.append((time.time() - start) * 1000)
    return latencies

def percentile(data, p):
    sorted_data = sorted(data)
    idx = int(len(sorted_data) * p / 100)
    return sorted_data[min(idx, len(sorted_data)-1)]

import random
random.seed(0)
def variable_latency_model(x):
    # simulate occasional slow requests (a realistic "long tail")
    if random.random() < 0.05:
        time.sleep(0.02)   # the rare slow request
    return x * 2

latencies = measure_latencies(variable_latency_model, range(200))
print(f"average latency: {sum(latencies)/len(latencies):.2f} ms")
print(f"p50 (median):     {percentile(latencies, 50):.2f} ms")
print(f"p95:               {percentile(latencies, 95):.2f} ms")
print(f"p99:               {percentile(latencies, 99):.2f} ms")

⚠️ **Common trap.** Reporting only the **average** latency hides tail problems. A system with
"average 50ms" might still make 1% of users wait 2 full seconds — always look at p95/p99, not
just the mean.

### ✏️ Your Turn 3.1
Compute throughput (requests/second) given that `latencies` (from above) has an **average**
latency in milliseconds — assuming requests are processed one at a time.

In [ ]:
avg_latency_ms = sum(latencies)/len(latencies)
throughput_per_sec = None
print(throughput_per_sec)

✅ **Solution**
```python
avg_latency_ms = sum(latencies)/len(latencies)
throughput_per_sec = 1000 / avg_latency_ms   # requests per second, single-threaded
```

---
## Chapter 4 — Caching Strategies

📖 **Theory.** Many production requests are **repeats** or near-repeats — caching avoids
recomputing (or re-calling an expensive LLM API for) the same answer. Common strategies:
- **Exact-match cache** — hash the exact input, return the cached output if seen before.
- **Semantic cache** — for LLM apps, cache by **similarity** (from the Embeddings lab!) so
  near-duplicate questions ("what's your refund policy" vs "how do refunds work") can share a
  cached answer.

🖼️ **Diagram — cache hit vs miss**
```
 request ─► hash/embed ─► check cache
                              │
                     hit ─────┴───── miss
                      │               │
              return cached      call the real model, THEN cache the result
              (fast, free)       (slow, costs money)
```


In [ ]:
class SimpleCache:
    def __init__(self, max_size=100):
        self.cache = {}
        self.max_size = max_size
        self.hits, self.misses = 0, 0
    def get_or_compute(self, key, compute_fn):
        if key in self.cache:
            self.hits += 1
            return self.cache[key]
        self.misses += 1
        result = compute_fn()
        if len(self.cache) >= self.max_size:
            self.cache.pop(next(iter(self.cache)))   # simple eviction: drop the oldest
        self.cache[key] = result
        return result
    def hit_rate(self):
        total = self.hits + self.misses
        return self.hits / total if total else 0.0

def expensive_llm_call(query):
    time.sleep(0.01)   # simulate API latency
    return f"answer to: {query}"

cache = SimpleCache()
queries = ["refund policy", "shipping time", "refund policy", "warranty", "refund policy"]
for q in queries:
    cache.get_or_compute(q, lambda q=q: expensive_llm_call(q))
print(f"hit rate: {cache.hit_rate():.0%}  (hits={cache.hits}, misses={cache.misses})")

⚡ **Pro tip.** Cache hit rate directly translates to cost savings for paid LLM APIs — a 40%
hit rate on a high-traffic FAQ bot can cut your API bill nearly in half.

### ✏️ Your Turn 4.1
Run the cache on a list of 10 queries where 7 are duplicates of just 2 unique questions. What
hit rate do you expect, and does the code confirm it?

In [ ]:
cache2 = SimpleCache()
queries2 = ["a","b","a","a","b","a","a","b","a","a"]
for q in queries2:
    cache2.get_or_compute(q, lambda q=q: expensive_llm_call(q))
print(cache2.hit_rate())

✅ **Solution**
```python
# 2 unique queries out of 10 total -> 2 misses, 8 hits -> hit rate 0.8 (80%)
```

---
## Chapter 5 — Monitoring: What to Track and Why

📖 **Theory.** You can't fix what you can't see. Core metrics to track for any served model:
- **Request volume** — how many calls, over time.
- **Latency** (p50/p95/p99, from Chapter 3).
- **Error rate** — fraction of failed requests.
- **Output distribution** — are predictions changing shape over time (a drift signal, next
  chapter)?

🖼️ **Diagram — a minimal monitoring dashboard**
```
 requests/min: 1,204     error_rate: 0.3%     p95 latency: 180ms
 prediction distribution: {billing: 40%, technical: 35%, shipping: 25%}   <- watch this shift over time
```


In [ ]:
class Monitor:
    def __init__(self):
        self.requests = []   # list of {timestamp, latency_ms, success, output}
    def log(self, latency_ms, success, output=None):
        self.requests.append({"ts": time.time(), "latency_ms": latency_ms, "success": success, "output": output})
    def summary(self):
        if not self.requests: return {}
        n = len(self.requests)
        errors = sum(1 for r in self.requests if not r["success"])
        latencies = [r["latency_ms"] for r in self.requests]
        outputs = Counter(r["output"] for r in self.requests if r["output"] is not None)
        return {
            "total_requests": n,
            "error_rate": round(errors/n, 4),
            "avg_latency_ms": round(sum(latencies)/n, 2),
            "p95_latency_ms": round(percentile(latencies, 95), 2),
            "output_distribution": dict(outputs),
        }

monitor = Monitor()
random.seed(1)
for i in range(50):
    success = random.random() > 0.05      # 5% failure rate
    latency = random.uniform(20, 200)
    output = random.choice(["billing","technical","shipping"]) if success else None
    monitor.log(latency, success, output)

print(json.dumps(monitor.summary(), indent=2))

### ✏️ Your Turn 5.1
Add 20 more log entries to `monitor` where the failure rate is artificially high (50%), then
print the updated summary. Does `error_rate` correctly reflect the mix of both batches?

In [ ]:
# log 20 more entries with a 50% failure rate, then print monitor.summary()


✅ **Solution**
```python
for i in range(20):
    success = random.random() > 0.5
    latency = random.uniform(20, 200)
    output = random.choice(["billing","technical","shipping"]) if success else None
    monitor.log(latency, success, output)
print(json.dumps(monitor.summary(), indent=2))
# error_rate rises, reflecting the blended 50 (5% fail) + 20 (50% fail) requests
```

---
## Chapter 6 — Data Drift & Model Drift

📖 **Theory.** Models degrade silently over time even without code changes:
- **Data drift** — the **input** distribution shifts (e.g. customers start asking about a new
  product category your training data never saw).
- **Model/concept drift** — the **relationship** between inputs and correct outputs changes (e.g.
  what counts as "urgent" shifts after a policy change), so even familiar inputs get worse
  answers.

🖼️ **Diagram — drift over time**
```
 training data:  [mostly billing & technical tickets]
 month 1 live:    [mostly billing & technical tickets]   <- matches training, healthy
 month 6 live:    [mostly SHIPPING tickets now]           <- drifted! model undertrained on this
```


In [ ]:
def distribution_distance(dist_a, dist_b):
    """A simple drift signal: total variation distance between two category distributions."""
    categories = set(dist_a) | set(dist_b)
    total_a, total_b = sum(dist_a.values()), sum(dist_b.values())
    diff = 0
    for c in categories:
        pa = dist_a.get(c, 0) / total_a if total_a else 0
        pb = dist_b.get(c, 0) / total_b if total_b else 0
        diff += abs(pa - pb)
    return diff / 2   # 0 = identical distributions, 1 = completely different

training_distribution = {"billing": 400, "technical": 350, "shipping": 250}
month1_live = {"billing": 380, "technical": 360, "shipping": 260}       # similar to training
month6_live = {"billing": 100, "technical": 150, "shipping": 750}       # big shift toward shipping

print("month1 drift score:", round(distribution_distance(training_distribution, month1_live), 3))
print("month6 drift score:", round(distribution_distance(training_distribution, month6_live), 3))
print("\n(higher score = more drift = model may need retraining/retuning on the new distribution)")

⚠️ **Common trap.** Drift can happen **without any error** being logged — the model still
returns confident predictions, just increasingly wrong ones for the new pattern of inputs. This
is why tracking the **input/output distribution** (not just error rate) matters.

### ✏️ Your Turn 6.1
Compute the drift score between `training_distribution` and a hypothetical live distribution
where the categories are **identical proportions**, just scaled up (e.g. `{"billing": 800,
"technical": 700, "shipping": 500}`). Confirm the drift score is ~0.

In [ ]:
proportional_scale = {"billing": 800, "technical": 700, "shipping": 500}
drift_score = None
print(drift_score)

✅ **Solution**
```python
drift_score = distribution_distance(training_distribution, proportional_scale)
# ~0.0 -- same PROPORTIONS, just more volume, so no real distribution shift
```

---
## Chapter 7 — A/B Testing & Canary Deployments

📖 **Theory.** Never fully switch to a new model version blindly. Two safety patterns:
- **A/B testing** — split traffic between the old (control) and new (variant) model, compare
  metrics, and only fully roll out if the variant genuinely performs better.
- **Canary deployment** — release the new version to a **small percentage** of traffic first
  (e.g. 5%), watch for problems, then gradually increase — limiting the "blast radius" if
  something's wrong.

🖼️ **Diagram — canary rollout**
```
 day 1:  5% traffic -> new model   95% -> old model     (small blast radius if broken)
 day 3:  25% traffic -> new model  75% -> old model      (looking healthy, expand)
 day 7:  100% traffic -> new model                        (fully rolled out)
```


In [ ]:
def route_traffic(request_id, canary_percent=10, seed_offset=0):
    # deterministic routing based on request_id -- same request always goes the same way
    bucket = (hash(str(request_id) + str(seed_offset)) % 100)
    return "canary" if bucket < canary_percent else "stable"

routing_counts = Counter(route_traffic(i, canary_percent=10) for i in range(1000))
print("traffic split over 1000 requests:", dict(routing_counts))
print(f"canary got ~{routing_counts['canary']/10:.0f}% of traffic (target: 10%)")

⚡ **Pro tip.** Route by a **stable hash of a user/request ID**, not randomly per-request —
this way the same user consistently gets the same version, avoiding a confusing experience where
behavior flips between requests.

### ✏️ Your Turn 7.1
Change `canary_percent` to 25 and confirm roughly 25% of 1000 simulated requests get routed to
`"canary"`.

In [ ]:
routing_counts_25 = None
print(routing_counts_25)

✅ **Solution**
```python
routing_counts_25 = Counter(route_traffic(i, canary_percent=25) for i in range(1000))
print(routing_counts_25)   # canary ~250, stable ~750
```

---
## Chapter 8 — Logging & Observability for LLM Apps

📖 **Theory.** LLM apps need **richer** logs than typical software: not just "request came in,
response went out" but the actual **prompt**, **response**, **token counts**, **latency**, and
**retrieved context** (for RAG) — because debugging "why did the model say that?" requires
seeing exactly what it was given.

🖼️ **Diagram — a rich LLM trace**
```
 trace_id: abc123
 ├─ prompt sent: "..." (with retrieved context attached)
 ├─ model: gpt-4o-mini, temperature: 0.2
 ├─ tokens: in=150, out=40
 ├─ latency: 340ms
 └─ response: "..."
```


In [ ]:
class LLMTracer:
    def __init__(self):
        self.traces = []
    def log_call(self, trace_id, prompt, response, model, tokens_in, tokens_out, latency_ms, context=None):
        self.traces.append({
            "trace_id": trace_id, "prompt": prompt, "response": response, "model": model,
            "tokens_in": tokens_in, "tokens_out": tokens_out, "latency_ms": latency_ms,
            "context": context, "timestamp": time.time(),
        })
    def find_slow_calls(self, threshold_ms=200):
        return [t for t in self.traces if t["latency_ms"] > threshold_ms]
    def total_tokens(self):
        return sum(t["tokens_in"] + t["tokens_out"] for t in self.traces)

tracer = LLMTracer()
tracer.log_call("t1", "What is the refund policy?", "Refunds within 30 days.",
                 "gpt-4o-mini", tokens_in=20, tokens_out=8, latency_ms=180,
                 context=["Our return window is 30 days..."])
tracer.log_call("t2", "Complex multi-part question...", "Detailed answer...",
                 "gpt-4o-mini", tokens_in=200, tokens_out=150, latency_ms=850)

print("total tokens used:", tracer.total_tokens())
print("slow calls (>200ms):", [t["trace_id"] for t in tracer.find_slow_calls()])

⚠️ **Common trap.** Logging the prompt/response is essential for debugging, but for
production systems handling real user data, this raises **privacy** concerns — apply the same
PII-redaction discipline from the Evaluation & Guardrails lab before persisting logs long-term.

### ✏️ Your Turn 8.1
Add a `log_call` for a third trace `"t3"` with `latency_ms=95` and confirm `find_slow_calls`
correctly excludes it (below the 200ms threshold).

In [ ]:
# log t3, then re-check find_slow_calls


✅ **Solution**
```python
tracer.log_call("t3", "quick question", "quick answer", "gpt-4o-mini",
                 tokens_in=10, tokens_out=5, latency_ms=95)
print([t["trace_id"] for t in tracer.find_slow_calls()])   # still just ["t2"]
```

---
## Chapter 9 — Cost Management at Scale

📖 **Theory.** At production volume, per-request costs (LLM API calls, compute) add up fast.
Key levers:
- **Caching** (Chapter 4) — avoid recomputing.
- **Model tiering** — use a cheap/fast model for easy cases, an expensive/accurate model only
  when needed (a simple router, like in the LangChain/Agents labs).
- **Batching** — group requests to amortize fixed overhead.
- **Budget alerts** — track spend in real time and alert before a runaway loop (recall the
  Agents lab's loop-safety chapter!) burns your budget.

🖼️ **Diagram — model tiering**
```
 request ─► classify difficulty ─┬─ easy   ──► cheap/fast model  ($0.0001/call)
                                  └─ hard   ──► expensive/accurate model  ($0.01/call)
                90% of traffic is often "easy" -- tiering can cut costs 10x+
```


In [ ]:
def estimate_difficulty(query):
    # a cheap heuristic: longer, more complex-looking queries are "harder"
    return "hard" if len(query.split()) > 15 or "?" in query[:-1] else "easy"

def tiered_cost(query, cheap_cost=0.0001, expensive_cost=0.01):
    tier = estimate_difficulty(query)
    cost = cheap_cost if tier == "easy" else expensive_cost
    return tier, cost

queries = [
    "refund status",
    "what is the return policy",
    "Can you explain in detail why my order was delayed and what compensation options exist given the circumstances?",
]
total_cost_tiered = 0
total_cost_flat = 0
for q in queries:
    tier, cost = tiered_cost(q)
    total_cost_tiered += cost
    total_cost_flat += 0.01   # if EVERY query used the expensive model
    print(f"'{q[:40]}...'  tier={tier}  cost=${cost}")

print(f"\ntotal cost with tiering: ${total_cost_tiered:.4f}")
print(f"total cost if always expensive: ${total_cost_flat:.4f}")
print(f"savings: {(1 - total_cost_tiered/total_cost_flat):.0%}")

### ✏️ Your Turn 9.1
Add a `budget_guard(current_spend, budget_limit)` function that returns `True` (allow the
request) only if `current_spend` is still under `budget_limit`, else `False` (block/alert).

In [ ]:
def budget_guard(current_spend, budget_limit):
    pass
print(budget_guard(45.0, 50.0))
print(budget_guard(55.0, 50.0))

✅ **Solution**
```python
def budget_guard(current_spend, budget_limit):
    return current_spend < budget_limit
```

---
## 🏆 Chapter 10 — Capstone: A Monitored, Guarded Serving Layer

Build a `ServingLayer` class combining: caching, monitoring, canary routing, tiered cost
estimation, and a budget guard — the production wrapper around any model you'd actually deploy.

In [ ]:
# Your ServingLayer here
class ServingLayer:
    def __init__(self, model_fn, budget_limit=10.0):
        pass
    def handle_request(self, request_id, query):
        pass
    def stats(self):
        pass

# serving = ServingLayer(lambda q: f"answer: {q}", budget_limit=1.0)
# print(serving.handle_request("req1", "refund status"))
# print(serving.stats())


✅ **Capstone Solution**
```python
class ServingLayer:
    def __init__(self, model_fn, budget_limit=10.0, canary_percent=10):
        self.model_fn = model_fn
        self.budget_limit = budget_limit
        self.canary_percent = canary_percent
        self.cache = SimpleCache(max_size=50)
        self.monitor = Monitor()
        self.spend = 0.0

    def handle_request(self, request_id, query):
        # 1. budget guard
        if not budget_guard(self.spend, self.budget_limit):
            return {"ok": False, "error": "budget_exceeded"}

        # 2. canary routing (illustrative -- doesn't change behavior here, just labels it)
        version = route_traffic(request_id, canary_percent=self.canary_percent)

        # 3. tiered cost estimate
        tier, cost = tiered_cost(query)

        # 4. cache + serve
        start = time.time()
        result = self.cache.get_or_compute(query, lambda: self.model_fn(query))
        latency_ms = (time.time() - start) * 1000
        self.spend += cost if latency_ms > 0.01 else 0   # only "charge" on a real (non-cached) compute, roughly

        # 5. monitor
        self.monitor.log(latency_ms, success=True, output=tier)

        return {"ok": True, "result": result, "version": version, "tier": tier, "latency_ms": round(latency_ms,3)}

    def stats(self):
        return {"monitor": self.monitor.summary(), "cache_hit_rate": self.cache.hit_rate(),
                "total_spend": round(self.spend, 4)}

serving = ServingLayer(lambda q: f"answer: {q}", budget_limit=1.0)
for q in ["refund status", "refund status", "what is the return policy", "shipping time"]:
    print(serving.handle_request(f"req-{q}", q))

print("\\nfinal stats:", json.dumps(serving.stats(), indent=2))
```

🎉 **You understand production MLOps!** What changes from notebook to production, serving
patterns, latency/throughput budgets, caching, monitoring, drift detection, safe rollout (A/B +
canary), rich LLM observability, and cost management. This is the operational discipline that
keeps a model reliable and affordable long after the notebook work is done.

---
### 📌 Concept Quick-Reference
**Notebook vs prod:** production needs its own error handling, scale, observability, safe rollout
**Serving patterns:** batch (scheduled), real-time (low latency), streaming (continuous)
**Latency:** track p95/p99, not just average — tail latency is the real user experience
**Caching:** exact-match or semantic; hit rate directly reduces cost
**Monitoring:** request volume, error rate, latency percentiles, output distribution
**Drift:** data drift (inputs shift) vs. model drift (input-output relationship shifts)
**Safe rollout:** A/B testing (compare) + canary (small % first, expand gradually)
**Observability:** log prompt/response/tokens/latency/context for LLM apps (redact PII)
**Cost:** caching + model tiering + batching + budget guards
